In [6]:
import os
import requests

FASTLY_API_HOST = "api.fastly.com"
FASTLY_API_TOKEN = "2wHRAMxgY5fjMxvlFdb3pUW61jXSiQi7"

def check_domain_availability(domain: str, scope: str = "estimate") -> dict:
    """
    Call Fastly Domain Research 'status' endpoint for a single domain.
    scope='estimate' is cheaper & good for 'is this available to register?' checks.
    Use scope=None or 'precise' for full registry-level checks.
    """
    url = f"https://{FASTLY_API_HOST}/domain-management/v1/tools/status"
    headers = {
        "Fastly-Key": FASTLY_API_TOKEN,
        "Accept": "application/json",
    }
    params = {"domain": domain}
    if scope:  # e.g. "estimate" or "precise"
        params["scope"] = scope

    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()


In [7]:
import string

tld = ".ml"  # Specify the TLD you want to check

# Generate all two-letter combinations (aa to zz)
base_names = [a + b for a in string.ascii_lowercase for b in string.ascii_lowercase]
domains = [f"{base}{tld}" for base in base_names]

print(f"Total domains generated: {len(domains)}")

Total domains generated: 676


In [9]:
import csv
import os

# Make sure the data directory exists
csv_file = "data/domains.csv"
os.makedirs(os.path.dirname(csv_file), exist_ok=True)

# Check which domains have already been processed
processed_domains = set()

if os.path.exists(csv_file):
    with open(csv_file, "r", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # Skip header if present
        for row in reader:
            if row:  # Make sure row is not empty
                processed_domains.add(row[0])  # domain is in first column

print(f"Already processed {len(processed_domains)} domains")

# Filter out already processed domains
remaining_domains = [d for d in domains if d not in processed_domains]
print(f"Remaining domains to process: {len(remaining_domains)}")

# Resume processing with append mode
with open(csv_file, "a", newline="") as f:
    writer = csv.writer(f)
    
    # Only write header if file is empty/new
    if len(processed_domains) == 0 and os.stat(csv_file).st_size == 0:
        writer.writerow(["domain", "zone", "status", "summary"])

    for i, domain in enumerate(remaining_domains):
        print(f"Processing {i+1}/{len(remaining_domains)}: {domain}")
        
        try:
            payload = check_domain_availability(domain) or {}

            # Fastly returns a single status object, not a list
            entry = payload

            writer.writerow([
                entry.get("domain", domain),
                entry.get("zone", ""),
                entry.get("status", ""),
                # Fastly doesn’t have "summary"; use tags or scope as a pseudo-summary
                entry.get("tags", entry.get("scope", "")),
            ])
            f.flush()
            
        except Exception as e:
            print(f"Error processing {domain}: {e}")
            continue


Already processed 0 domains
Remaining domains to process: 676
Processing 1/676: aa.ml
Processing 2/676: ab.ml
Processing 3/676: ac.ml
Processing 4/676: ad.ml
Processing 5/676: ae.ml
Processing 6/676: af.ml
Processing 7/676: ag.ml
Processing 8/676: ah.ml
Processing 9/676: ai.ml
Processing 10/676: aj.ml
Processing 11/676: ak.ml
Processing 12/676: al.ml
Processing 13/676: am.ml
Processing 14/676: an.ml
Processing 15/676: ao.ml
Processing 16/676: ap.ml
Processing 17/676: aq.ml
Processing 18/676: ar.ml
Processing 19/676: as.ml
Processing 20/676: at.ml
Processing 21/676: au.ml
Processing 22/676: av.ml
Processing 23/676: aw.ml
Processing 24/676: ax.ml
Processing 25/676: ay.ml
Processing 26/676: az.ml
Processing 27/676: ba.ml
Processing 28/676: bb.ml
Processing 29/676: bc.ml
Processing 30/676: bd.ml
Processing 31/676: be.ml
Processing 32/676: bf.ml
Processing 33/676: bg.ml
Processing 34/676: bh.ml
Processing 35/676: bi.ml
Processing 36/676: bj.ml
Processing 37/676: bk.ml
Processing 38/676: bl.

In [5]:
import csv, os

csv_file = "data/domains.csv"
rows = {}  # domain -> dict(zone,status,summary)

# Load existing rows (if any)
if os.path.exists(csv_file):
    with open(csv_file, "r", newline="") as f:
        r = csv.reader(f)
        header = next(r, None)
        for row in r:
            if not row: 
                continue
            dom = row[0].strip()
            rows[dom] = {
                "zone": row[1].strip() if len(row) > 1 else "",
                "status": row[2].strip().lower() if len(row) > 2 else "",
                "summary": row[3].strip() if len(row) > 3 else "",
            }

ERROR_STATUSES = {"", "error", "failed", "timeout", "unknown"}

# Figure out which to (re)process
to_fix = [d for d in domains if d not in rows or rows[d]["status"] in ERROR_STATUSES]
print(f"Retrying {len(to_fix)} domains (missing or previously errored)…")

# Re-check and update in-memory rows
for d in to_fix:
    try:
        payload = check_domain_availability(d) or {}
        entries = payload.get("status", [])
        # Use the first matching entry (fallbacks if API returns nothing)
        e = next((e for e in entries if e.get("domain") == d), entries[0] if entries else {})
        rows[d] = {
            "zone": e.get("zone", ""),
            "status": (e.get("status", "") or "").lower(),
            "summary": e.get("summary", ""),
        }
    except Exception as ex:
        rows[d] = {"zone": "", "status": "error", "summary": f"retry_failed: {ex}"}

# Write back a clean CSV (header + all domains)
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
with open(csv_file, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["domain", "zone", "status", "summary"])
    for d in sorted(rows.keys()):
        r = rows[d]
        w.writerow([d, r["zone"], r["status"], r["summary"]])

print("Done.")


Retrying 0 domains (missing or previously errored)…
Done.


In [6]:
# Print any that still failed after retry
still_bad = [d for d, r in rows.items() if r["status"] in ERROR_STATUSES]
print(f"{len(still_bad)} domains still failed after retry.")

if still_bad:
    print("These domains still failed:", still_bad)
else:
    print("✅ All domains processed successfully")


0 domains still failed after retry.
✅ All domains processed successfully
